# Nemotron Reasoning Challenge — training notebook

Clean pipeline matching `scripts/` on branch `build/nemotron-pipeline`.

**Before running:** Add Input → (1) the **competition data**, (2) the model **`nemotron-3-nano-30b-a3b-bf16`** (publisher `metric`). Internet **ON** is simplest. See `WRITEUP.md` for method; `kaggle_wheels/README.md` for the internet-OFF path.

## 1. Get the code

In [ ]:
!rm -rf repo && git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!ls scripts

## 2. Core dependencies (internet ON) + availability check

In [ ]:
!pip install -q peft trl datasets accelerate bitsandbytes
import importlib
for m in ["torch","transformers","peft","trl","datasets","accelerate","bitsandbytes","mamba_ssm","causal_conv1d"]:
    try: importlib.import_module(m); print('ok  ', m)
    except Exception as e: print('MISS', m, '->', repr(e)[:80])

## 2b. Install mamba_ssm + causal_conv1d if MISSING (internet ON)
Grabs prebuilt wheels matching this exact torch/CUDA. Skip if step 2 showed both `ok`.

In [ ]:
!python kaggle_wheels/fetch_torch_locked_wheels.py --dest /kaggle/working/mw
!pip install --no-index --no-deps /kaggle/working/mw/causal_conv1d-*.whl /kaggle/working/mw/mamba_ssm-*.whl
import importlib
for m in ("causal_conv1d","mamba_ssm"):
    try: importlib.import_module(m); print('ok ', m)
    except Exception as e: print('FAIL', m, '->', repr(e)[:120])

## 3. Bring in the competition data (recursive find)

In [ ]:
import glob, os, shutil
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, "train.csv not found — Add Input -> the competition"
shutil.copy(hits[0], 'data/train.csv'); print('train.csv <-', hits[0])

## 4. EDA + build the SFT data (no GPU)

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data   # -> data/train_sft.jsonl

## 5. Train the LoRA adapter (2xT4)
8-bit + CPU offload; a smoke test runs first. On OOM the script writes **`FALLBACK.md`** — move this one step to a rented A100/H100 and upload the resulting `lora_adapter/` as a dataset.

In [ ]:
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir /kaggle/working/lora_adapter

## 6. (Optional) Local eval — needs vLLM

In [ ]:
!python scripts/04_evaluate.py --adapter-path /kaggle/working/lora_adapter --data-dir data

## 7. Package the submission

In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip